In [ ]:
import torch
import torch.nn as nn  # neural net layers and activations
from torch.optim import SGD  # Our chosen optimizer
from torch.utils.data import DataLoader, TensorDataset  # Super useful data utilities!

from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Used for visualization
import torchvision.utils as vutils
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

# Turn off some annoying convergence warnings from sklearn
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

from IPython.display import display, HTML

display(HTML("""
<style>
div.output pre {
    white-space: pre-wrap;      /* Wrap long lines */
    word-break: break-word;     /* Break words if necessary */
}
</style>
"""))  # prevent long lines from making unreadable PDF subs

KeyboardInterrupt: 

## **Obtain and inspect data**


In [ ]:
# Load dataset and visualize
import os
if not os.path.exists('data.pt'):
    raise FileNotFoundError("data.pt not found! Please download the dataset and place it in the project root directory.")

X, y = torch.load('data.pt')

# Validate data shapes
assert X.shape[0] == y.shape[0], "Number of samples in X and y must match"
assert len(X.shape) == 4, "X should be 4D tensor (N, C, H, W)"
assert X.shape[1] == 3, "Expected 3 color channels"
assert X.shape[2] == 32 and X.shape[3] == 32, "Expected 32x32 images"

print('Data shapes before flattening:')
print('X:', X.shape)  # 2000, 3, 32, 32, 2000 images, channel, height width
print('y:', y.shape)  # 2000 binary labels 0 is real, 1 is fake

# Print examples from each class
# Ensure tensors are on CPU for visualization
grid = vutils.make_grid(X[y==0][:8].cpu(), nrow=4, padding=2, normalize=True)
fig, axs = plt.subplots(2, 1, figsize=(8, 8))
axs[0].axis('off')
axs[0].set_title('REAL Cat images')
axs[0].imshow(grid.numpy().transpose((1, 2, 0)))

grid = vutils.make_grid(X[y==1][:8].cpu(), nrow=4, padding=2, normalize=True)
axs[1].axis('off')
axs[1].set_title('FAKE Cat images')
axs[1].imshow(grid.numpy().transpose((1, 2, 0)))


X = X.flatten(start_dim=1)  # From now on, we work with the flattened vector
print(f"X shape after flattening: {X.shape}\n")

NameError: name 'torch' is not defined

In [ ]:
# Use scikit-learn logistic regression (with default hyper-parameters) 
# with 5-fold CV to get the train and validation accuracies
# for a simple linear classifier - a good baseline for our MLP

# Constants for cross-validation
N_FOLDS = 5
RANDOM_STATE = 42

val_accs = []  # store validation accuracy for each fold
train_accs = []  # store training accuracy for each fold

# iterate over folds, 
# remember to use "shuffle=True", as datapoints are not shuffled
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # Compute and store accuracy on train data
    train_acc = model.score(X_train, y_train)
    train_accs.append(train_acc)
    
    # Compute and store accuracy on validation data
    val_acc = model.score(X_val, y_val)
    val_accs.append(val_acc)

train_std, train_mean = torch.std_mean(torch.tensor(train_accs))
val_std, val_mean = torch.std_mean(torch.tensor(val_accs))

# Standard error is standard deviation / sqrt(n), it is more typical to report this
rootn = torch.sqrt(torch.tensor(N_FOLDS))  # n is number of folds
train_se = train_std / rootn
val_se = val_std / rootn

print(f'Train Accuracy and standard error:\t {train_mean:.3f} +/- {train_se:.3f}')
print(f'Validation Accuracy and standard error:\t {val_mean:.3f} +/- {val_se:.3f}')

Train Accuracy and standard error:	 0.991 +/- 0.001
Validation Accuracy and standard error:	 0.634 +/- 0.011


## Defining a Multi-Layer Perceptron

- Implement an __init__ function and a forward function
- Use Linear layers with ReLU activations for the hidden layers
- 2 layers of hidden units. First layer has 128 hidden units, second layer has 64 hidden units. 
- Output must have one logit for each of the two classes

In [ ]:
class MyMLP(nn.Module):
    # Define a multilayer perceptron. Criteria above
    def __init__(self, n_hidden=128, n_hidden2=64):
        super(MyMLP, self).__init__()
        
        # Input dimensions are 3072 (flattened 3x32x32 images)
        self.layer1 = nn.Linear(3072, n_hidden)
        self.layer2 = nn.Linear(n_hidden, n_hidden2)
        self.layer3 = nn.Linear(n_hidden2, 2)  # 2 output classes (real/fake)
        
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = self.layer3(x)  # No activation on output layer (raw logits)
        return x


## Train function

In [ ]:

def train(model, train_loader, val_loader, n_epochs, optimizer, loss_fn, verbose=False, patience=5):
    """
    Train model using data from train_loader over n_epochs.
    
    Args:
        model: PyTorch model to train
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        n_epochs: Number of training epochs
        optimizer: PyTorch optimizer (e.g., SGD)
        loss_fn: Loss function (e.g., CrossEntropyLoss)
        verbose: If True, print training progress
        patience: Number of epochs to wait before early stopping
    
    Returns:
        None (model is trained in-place)
    """
    # Early stopping variables
    best_val_loss = float('inf')
    counter = 0
    
    for epoch in range(n_epochs):
        # Train loop
        model.train()  # Set model to training mode
        train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(batch_x)
            loss = loss_fn(outputs, batch_y)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_x.size(0)
        
        # Calculate average loss over training set
        train_loss = train_loss / len(train_loader.dataset)
        
        # Always do validation to check for early stopping
        model.eval()  # Set model to evaluation mode
        val_loss = 0.0
        
        with torch.no_grad():  # Disable gradient computation for validation
            for batch_x, batch_y in val_loader:
                outputs = model(batch_x)
                loss = loss_fn(outputs, batch_y)
                val_loss += loss.item() * batch_x.size(0)
        
        # Calculate average loss over validation set
        val_loss = val_loss / len(val_loader.dataset)
        
        if verbose:
            # Print out train/val loss during development
            print(f'Epoch {epoch+1}/{n_epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}')
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Set model to evaluation mode after training
    model.eval()

Loop over hyper-parameters and do 5-fold cross-validation for each setting, saving the train and validation mean accuracy and standard error.

In [ ]:

# Hyperparameter configuration
LEARNING_RATES = [0.001, 0.01, 0.1]
WEIGHT_DECAYS = [0.0, 0.01]
BATCH_SIZE = 50
N_EPOCHS = 20
N_FOLDS_CV = 5

results = []  # store results for each hyper-parameter
for lr in LEARNING_RATES:
    for wd in WEIGHT_DECAYS: 
        val_accs = []  # store validation accuracy for each fold
        train_accs = []  # store training accuracy for each fold
        # iterate over folds, with "shuffle=True"
        kf = KFold(n_splits=N_FOLDS_CV, shuffle=True, random_state=RANDOM_STATE)
        
        for train_idx, val_idx in kf.split(X):
            # Split data into train and validation
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            # Create data loaders to pass to training loop
            train_dataset = TensorDataset(X_train, y_train)
            val_dataset = TensorDataset(X_val, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

            # Initialize model, loss_fn (Cross entropy loss), 
            # and optimizer (SGD with various hyperparameters)
            model = MyMLP()
            loss_fn = nn.CrossEntropyLoss()
            optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd)
            
            # Call your training function
            train(model, train_loader, val_loader, n_epochs, optimizer, loss_fn, verbose=False, patience=3)
            
            with torch.no_grad():
                # Use the trained model to estimate train/val accuracy
                
                # Calculate training accuracy
                train_outputs = model(X_train)
                _, train_preds = torch.max(train_outputs, 1)
                train_correct = (train_preds == y_train).sum().item()
                train_acc = train_correct / len(y_train)
                train_accs.append(train_acc)

                # Calculate validation accuracy
                val_outputs = model(X_val)
                _, val_preds = torch.max(val_outputs, 1)
                val_correct = (val_preds == y_val).sum().item()
                val_acc = val_correct / len(y_val)
                val_accs.append(val_acc)

        # For each hyper-parameter, I'm storing the parameter values and 
        # the mean and standard error of accuracy in a list in "results". 
        train_std, train_mean = torch.std_mean(torch.tensor(train_accs))
        val_std, val_mean = torch.std_mean(torch.tensor(val_accs))
        rootn = torch.sqrt(torch.tensor(n_folds))  # n is number of folds
        train_se, val_se = train_std / rootn, val_std / rootn
        # Storing learning rate, weight decay value, train mean accuracy, standard error, val mean accuracy, standard error 
        results.append((lr, wd, train_mean.item(), train_se.item(), val_mean.item(), val_se.item()))

## Show result 

In [ ]:

# Create a DataFrame from the list of tuples, with labeled columns
column_names = ['learning_rate', 'weight_decay', 'train_mean', 'train_se','val_mean', 'val_se']
df = pd.DataFrame(results, columns=column_names)

df['train_output'] = df.apply(lambda row: f"{row['train_mean']:.3f} +/- {row['train_se']:.3f}", axis=1)
df['val_output'] = df.apply(lambda row: f"{row['val_mean']:.3f} +/- {row['val_se']:.3f}", axis=1)

print('Training results')
pivot_df = df.pivot(index='weight_decay', columns='learning_rate', values='train_output')
display(pivot_df)

print('Validation results')
pivot_df = df.pivot(index='weight_decay', columns='learning_rate', values='val_output')
display(pivot_df)

Training results


learning_rate,0.001,0.010
weight_decay,,
0.00,0.694 +/- 0.034,0.894 +/- 0.003
0.01,0.694 +/- 0.011,NaN


Validation results


learning_rate,0.001,0.010
weight_decay,,
0.00,0.656 +/- 0.028,0.804 +/- 0.007
0.01,0.659 +/- 0.015,NaN
